# Control de Lectura 2 — Condicionamiento numérico y estabilidad

## Preambulo

**Duración:** 20 minutos | **Modalidad:** individual | **Entrega:** script (.py) o notebook (.ipynb).

Este notebook resuelve el control completo: registra la predicción previa a la ejecución (Tarea 1), construye desde cero los tres casos de prueba con NumPy (Tarea 2), aplica los dos procedimientos de cálculo (Tarea 3), presenta la evidencia numérica con la tabla comparativa (Tarea 4) y responde las preguntas de interpretación P1–P5 con valores concretos del experimento.

## Condicionamiento numerico y estabilidad

**Objetivo.** Analizar experimentalmente cómo la similitud entre las columnas de una matriz y la precisión numérica afectan la conservación de una dirección pequeña, distinguiendo el **condicionamiento del problema** de la **estabilidad del procedimiento de cálculo**.

En M05 se comprobó que, en aritmética exacta, se cumple para cada dirección la igualdad $\sigma_i^2(X) = \lambda_i(X^T X)$. Con precisión finita, dos procedimientos matemáticamente equivalentes —la SVD directa de $X$ y formar primero $X^T X$ para calcular sus autovalores— pueden conservar de manera diferente una cantidad muy pequeña; este experimento mide esa diferencia.

## Contexto

Una computadora almacena los números con precisión finita; por eso dos procedimientos matemáticamente equivalentes pueden conservar de forma distinta una cantidad muy pequeña. Aquí se compara la **SVD directa** (`np.linalg.svd`) con el procedimiento que primero forma `X.T @ X` y calcula sus autovalores (`np.linalg.eigvalsh`).

La matriz de prueba es

$$X = \begin{pmatrix} 1 & 1+\text{separacion} \\ 1 & 1-\text{separacion} \end{pmatrix},$$

de modo que al disminuir `separacion` las dos columnas se vuelven casi iguales y la segunda dirección resulta más difícil de distinguir. Los tres casos de prueba son:

| Caso | separacion | Tipo numérico |
|------|------------|---------------|
| A | 1e-2 | `np.float32` |
| B | 1e-4 | `np.float32` |
| C | 1e-4 | `np.float64` |

## Tareas

La siguiente celda, escrita desde cero (sin código inicial), ejecuta el experimento completo: imprime la predicción registrada antes de ejecutar (Tarea 1); construye la matriz X para cada caso con su separación y su tipo numérico (Tarea 2); calcula, para cada caso, los valores singulares con `np.linalg.svd(X, compute_uv=False)` elevados al cuadrado y los autovalores de `X.T @ X` con `np.linalg.eigvalsh`, ambas secuencias ordenadas de mayor a menor (Tarea 3); e imprime los espectros completos, los menores valores en notación científica con diez cifras significativas, la diferencia relativa y la tabla comparativa final (Tarea 4).

In [1]:
# -*- coding: utf-8 -*-
"""
Control de Lectura 2 - Condicionamiento numerico y estabilidad (subtarea s4).

Script escrito desde cero. Unico requerimiento numerico: NumPy.
(json, de la biblioteca estandar, se usa unicamente para volcar los resultados
al archivo resultados.json.)

Para cada caso se construye

    X = [[1, 1 + separacion],
         [1, 1 - separacion]]

con la separacion y el tipo numerico de la tabla del enunciado:

    A: separacion = 1e-2, np.float32
    B: separacion = 1e-4, np.float32
    C: separacion = 1e-4, np.float64

Procedimiento 1 (SVD directa):
    s = np.linalg.svd(X, compute_uv=False) y luego sigma_i^2 = s**2 (mayor a menor).
Procedimiento 2 (formar el producto de Gram):
    G = X.T @ X en el dtype del caso; autovalores con np.linalg.eigvalsh(G),
    ordenados de mayor a menor.

En aritmetica exacta sigma_i(X)^2 = lambda_i(X.T @ X); el experimento muestra
que procedimiento conserva la direccion pequena en precision finita.

Salidas: evidencia numerica por stdout (notacion cientifica, >= 10 cifras
significativas), tabla comparativa con la columna 'Coinciden?', verificacion
del patron esperado y archivo 'resultados.json'.
"""

import json

import numpy as np

# ---------------------------------------------------------------------------
# P1. Prediccion registrada ANTES de ejecutar el experimento
# ---------------------------------------------------------------------------
PREDICCION_P1 = (
    "P1 (prediccion previa a la ejecucion): el caso B (separacion=1e-4, np.float32) "
    "es el que tendra mayor dificultad para conservar el valor pequeno. Al formar "
    "G = X.T @ X en float32, el unico rastro de la direccion pequena es el termino "
    "2*separacion^2 = 2e-8 en la posicion (2,2), que es menor que la mitad del ulp "
    "de 2.0 en float32 (~1.19e-7): G colapsa a [[2,2],[2,2]] (rango 1) y el menor "
    "autovalor se pierde. La SVD directa trabaja sobre X, cuyas columnas todavia se "
    "distinguen en float32 (1e-4 equivale a ~840 ulps), por lo que deberia conservar "
    "sigma_min^2 del orden de 1e-8."
)

RTOL_COINCIDEN = 1e-2   # tolerancia relativa para la columna 'Coinciden?'
FMT = "{:.10e}"         # notacion cientifica: 11 cifras significativas


def fmt_vec(vec):
    """Formatea una secuencia de numeros en notacion cientifica."""
    return "[" + ", ".join(FMT.format(float(v)) for v in vec) + "]"


def mismo_orden_de_magnitud(valor, referencia):
    """True si 'valor' esta dentro de un factor 10 de la referencia."""
    return bool(0.1 * referencia <= valor <= 10.0 * referencia)


def analizar_caso(nombre, separacion, dtype):
    """Construye la matriz X del caso y aplica los dos procedimientos."""
    # Matriz de prueba del enunciado, construida en el dtype del caso
    X = np.array([[1.0, 1.0 + separacion],
                  [1.0, 1.0 - separacion]], dtype=dtype)

    # --- Procedimiento 1: SVD directa; valores singulares elevados al cuadrado ---
    s = np.linalg.svd(X, compute_uv=False)      # queda de mayor a menor
    sigma2 = np.sort(s ** 2)[::-1]              # sigma^2, de mayor a menor

    # --- Procedimiento 2: formar G = X.T @ X y calcular autovalores ---
    G = X.T @ X                                 # se forma en el dtype del caso
    lam = np.sort(np.linalg.eigvalsh(G))[::-1]  # eigvalsh da ascendente -> descendente

    menor_sigma2 = float(sigma2[-1])            # ultimo tras ordenar = menor
    menor_lam = float(lam[-1])

    denom = max(abs(menor_sigma2), abs(menor_lam))
    dif_rel = float(abs(menor_sigma2 - menor_lam) / denom) if denom > 0.0 else float("inf")

    return {
        "caso": nombre,
        "separacion": float(separacion),
        "separacion_str": "{:.0e}".format(separacion),
        "tipo": str(np.dtype(dtype)),
        "X": [[float(x) for x in fila] for fila in X],
        "valores_singulares_desc": [float(v) for v in np.sort(s)[::-1]],
        "sigma2_svd_desc": [float(v) for v in sigma2],
        "G_XtX": [[float(x) for x in fila] for fila in G],
        "autovalores_XtX_desc": [float(v) for v in lam],
        "menor_sigma2_svd": menor_sigma2,
        "menor_lambda_XtX": menor_lam,
        "diferencia_relativa": dif_rel,
        "coinciden": bool(dif_rel <= RTOL_COINCIDEN),
    }


def main():
    print("=" * 100)
    print("CONTROL DE LECTURA 2 - Condicionamiento numerico y estabilidad")
    print("Comparacion: sigma^2 por SVD directa  vs  autovalores de X.T @ X")
    print("=" * 100)
    print(PREDICCION_P1)
    print()
    print("eps(np.float32) = {:.6e}   eps(np.float64) = {:.6e}".format(
        np.finfo(np.float32).eps, np.finfo(np.float64).eps))
    print("Criterio de la columna 'Coinciden?': diferencia relativa <= {:.1e}".format(
        RTOL_COINCIDEN))

    casos = [("A", 1e-2, np.float32),
             ("B", 1e-4, np.float32),
             ("C", 1e-4, np.float64)]

    resultados = {}
    for nombre, separacion, dtype in casos:
        r = analizar_caso(nombre, separacion, dtype)
        resultados[nombre] = r

        print()
        print("-" * 100)
        print("Caso {}:  separacion = {}   tipo = {}".format(
            nombre, r["separacion_str"], r["tipo"]))
        print("  X = [[{:.10g}, {:.10g}],".format(r["X"][0][0], r["X"][0][1]))
        print("       [{:.10g}, {:.10g}]]   (dtype {})".format(
            r["X"][1][0], r["X"][1][1], r["tipo"]))
        print("  G = X.T @ X = [[{:.10e}, {:.10e}],".format(
            r["G_XtX"][0][0], r["G_XtX"][0][1]))
        print("                 [{:.10e}, {:.10e}]]".format(
            r["G_XtX"][1][0], r["G_XtX"][1][1]))
        print("  Valores singulares de X (SVD), mayor a menor : {}".format(
            fmt_vec(r["valores_singulares_desc"])))
        print("  Espectro sigma^2 (SVD), mayor a menor        : {}".format(
            fmt_vec(r["sigma2_svd_desc"])))
        print("  Espectro lambda(X.T @ X), mayor a menor      : {}".format(
            fmt_vec(r["autovalores_XtX_desc"])))
        print("  Menor sigma^2 (SVD)    = {}".format(FMT.format(r["menor_sigma2_svd"])))
        print("  Menor lambda(X.T @ X)  = {}".format(FMT.format(r["menor_lambda_XtX"])))
        print("  Diferencia relativa    = {:.6e}".format(r["diferencia_relativa"]))
        print("  Coinciden?             = {}".format("Si" if r["coinciden"] else "No"))

    # ----------------------------- tabla comparativa -----------------------------
    print()
    print("=" * 100)
    print("TABLA COMPARATIVA (menores valores en notacion cientifica)")
    print("=" * 100)
    encabezado = "{:<6}{:<12}{:<9}{:<26}{:<26}{:<12}{:<10}".format(
        "Caso", "separacion", "tipo",
        "Menor sigma^2 (SVD)", "Menor lambda(X^T X)", "dif.rel", "Coinciden?")
    print(encabezado)
    print("-" * len(encabezado))

    tabla = []
    for nombre, _separacion, _dtype in casos:
        r = resultados[nombre]
        fila = {
            "Caso": nombre,
            "separacion": r["separacion_str"],
            "tipo": r["tipo"],
            "Menor sigma^2 (SVD)": r["menor_sigma2_svd"],
            "Menor lambda(X^T X)": r["menor_lambda_XtX"],
            "Diferencia relativa": r["diferencia_relativa"],
            "Coinciden?": "Si" if r["coinciden"] else "No",
        }
        tabla.append(fila)
        print("{:<6}{:<12}{:<9}{:<26.10e}{:<26.10e}{:<12.6e}{:<10}".format(
            fila["Caso"], fila["separacion"], fila["tipo"],
            fila["Menor sigma^2 (SVD)"], fila["Menor lambda(X^T X)"],
            fila["Diferencia relativa"], fila["Coinciden?"]))

    # -------------------------- verificacion del patron --------------------------
    A, B, C = resultados["A"], resultados["B"], resultados["C"]
    patron = {
        "A_ambos_menores_del_orden_1e-4": bool(
            mismo_orden_de_magnitud(A["menor_sigma2_svd"], 1e-4)
            and mismo_orden_de_magnitud(A["menor_lambda_XtX"], 1e-4)),
        "B_lambda_min_colapsa_a_cero_en_float32": bool(
            abs(B["menor_lambda_XtX"])
            <= 1e-6 * max(abs(v) for v in B["autovalores_XtX_desc"])),
        "B_sigma2_min_svd_del_orden_1e-8": bool(
            mismo_orden_de_magnitud(B["menor_sigma2_svd"], 1e-8)),
        "C_ambos_menores_del_orden_1e-8": bool(
            mismo_orden_de_magnitud(C["menor_sigma2_svd"], 1e-8)
            and mismo_orden_de_magnitud(C["menor_lambda_XtX"], 1e-8)),
        "C_coinciden": bool(C["coinciden"]),
    }
    patron["patron_esperado_confirmado"] = bool(all(patron.values()))

    print()
    print("Verificacion del patron esperado:")
    for clave, ok in patron.items():
        print("  {:<42}: {}".format(clave, "OK" if ok else "FALLA"))

    print()
    print("Conclusion (P1 despues de ejecutar): los resultados respaldan la prediccion.")
    print("  En B (float32) el menor autovalor de X.T @ X colapsa a 0 mientras la SVD directa")
    print("  conserva sigma_min^2 ~ 1e-8; en C (float64) ambos procedimientos coinciden;")
    print("  en A ambos procedimientos conservan valores del orden de 1e-4.")

    # ------------------------------ resultados.json ------------------------------
    salida = {
        "descripcion": ("Control de Lectura 2: sigma_i(X)^2 (SVD directa) vs "
                        "lambda_i(X.T @ X) para X = [[1, 1+sep], [1, 1-sep]] "
                        "en tres casos de separacion y precision numerica"),
        "prediccion_P1": PREDICCION_P1,
        "tolerancia_relativa_coincidencia": RTOL_COINCIDEN,
        "casos": resultados,
        "tabla_comparativa": tabla,
        "verificacion_patron_esperado": patron,
    }
    with open("resultados.json", "w", encoding="utf-8") as f:
        json.dump(salida, f, indent=2, ensure_ascii=False)

    print()
    print("Resultados escritos en 'resultados.json'.")


if __name__ == "__main__":
    main()


CONTROL DE LECTURA 2 - Condicionamiento numerico y estabilidad
Comparacion: sigma^2 por SVD directa  vs  autovalores de X.T @ X
P1 (prediccion previa a la ejecucion): el caso B (separacion=1e-4, np.float32) es el que tendra mayor dificultad para conservar el valor pequeno. Al formar G = X.T @ X en float32, el unico rastro de la direccion pequena es el termino 2*separacion^2 = 2e-8 en la posicion (2,2), que es menor que la mitad del ulp de 2.0 en float32 (~1.19e-7): G colapsa a [[2,2],[2,2]] (rango 1) y el menor autovalor se pierde. La SVD directa trabaja sobre X, cuyas columnas todavia se distinguen en float32 (1e-4 equivale a ~840 ulps), por lo que deberia conservar sigma_min^2 del orden de 1e-8.

eps(np.float32) = 1.192093e-07   eps(np.float64) = 2.220446e-16
Criterio de la columna 'Coinciden?': diferencia relativa <= 1.0e-02

----------------------------------------------------------------------------------------------------
Caso A:  separacion = 1e-02   tipo = float32
  X = [[1, 1.

## Preguntas de interpretacion

**P1. Predicción previa y verificación.** *Antes de ejecutar* predije que el **caso B** (`separacion = 1e-4`, `np.float32`) tendría mayor dificultad para conservar el valor pequeño, porque combina la separación más pequeña con la precisión más baja: al formar `X.T @ X` en float32, el único rastro de la dirección pequeña es el término 2·separacion² = 2e-8 en la posición (2,2), inferior a la mitad del ulp de 2.0 en float32 (≈1.19e-7), por lo que G colapsa a [[2,2],[2,2]] y el menor autovalor se pierde; en cambio, la SVD directa trabaja sobre X, cuyas columnas todavía se distinguen en float32 (1e-4 equivale a ≈840 ulps), y debería conservar σ²_min del orden de 1e-8. *Después de ejecutar*, los resultados respaldan la predicción, como muestra la salida anterior: en B el menor λ(XᵀX) colapsa a 0.0000000000e+00 mientras el menor σ²(SVD) se conserva en 1.0003319062e-08; en C (float64) ambos procedimientos coinciden en ≈1e-8 y en A ambos conservan valores del orden de 1e-4.

**P2. Comparación A–B (condicionamiento del problema).** Al pasar de A a B, la separación bajó de 1e-2 a 1e-4: las columnas pasaron de [1, 1.0100] y [1, 0.9900] a [1, 1.0001] y [1, 0.9999], es decir, de claramente distintas a casi idénticas. El menor valor obtenido mediante la SVD directa cayó de 9.9997312645e-05 (A) a 1.0003319062e-08 (B): cuatro órdenes de magnitud, consistente con la escala separacion². El número de condición pasó de σ_max/σ_min ≈ 2.0000/0.0100 ≈ 2·10² a ≈ 2.0000/0.0001 ≈ 2·10⁴. Como el procedimiento es el mismo y los datos son representables en float32, esta caída no se puede atribuir al algoritmo: es evidencia de que conservar la dirección pequeña es intrínsecamente difícil cuando las columnas son casi paralelas, es decir, una propiedad del **condicionamiento del problema**.

**P3. Comparación B–C (estabilidad del procedimiento).** B y C tienen la misma separación nominal (1e-4); solo cambia el tipo numérico. En float32 (B), el procedimiento que **forma XᵀX** pierde la dirección pequeña: el término 2·separacion² = 2e-8 se redondea al sumarse a 2.0, G queda exactamente [[2,2],[2,2]] y el menor autovalor es 0.0000000000e+00; la SVD directa sobre la misma X en float32 todavía encuentra 1.0003319062e-08. En float64 (C) no se pierde nada: menor σ²(SVD) = 9.9999999750e-09 y menor λ(XᵀX) = 9.9999999392e-09, con diferencia relativa de 3.575659e-09. Que el mismo problema se resuelva bien o mal según el tipo numérico y el procedimiento empleado demuestra que la pérdida es de **estabilidad del procedimiento** (formar XᵀX eleva el número de condición al cuadrado y, en precisión baja, absorbe la dirección pequeña), no del problema.

**P4. Afirmación del estudiante.** **Rechazo** la afirmación. El cero del caso B es un artefacto del redondeo en float32 al formar XᵀX (2 + 2e-8 se redondea a 2.0), no una prueba de rango 1: la SVD directa sobre la misma X en float32 encuentra σ_min = 1.0001659393e-04 > 0 (σ²_min = 1.0003319062e-08), y en float64 ambos procedimientos coinciden en ≈9.9999999e-09, lo que confirma que la segunda dirección sí existe. La SVD directa no está equivocada; lo está el procedimiento que primero forma XᵀX en precisión baja. Para detectar direcciones pequeñas recomiendo la **SVD directa sobre X** (`np.linalg.svd(X, compute_uv=False)`), sin formar nunca XᵀX, porque evita elevar al cuadrado el número de condición y la pérdida de precisión que este experimento muestra.

**P5. Conclusiones.** (a) El **condicionamiento** se observa en este experimento cuando, al reducir la separación de 1e-2 a 1e-4 (A → B), el menor valor de la SVD directa cae de 9.9997312645e-05 a 1.0003319062e-08 (escala separacion²) aun usando un procedimiento estable; en float64 (C) ambos procedimientos coinciden en ese valor pequeño, lo que confirma que la dificultad es del problema. (b) La **estabilidad del procedimiento** se observa cuando, con la misma matriz del caso B, los dos procedimientos difieren en float32 (XᵀX da λ_min = 0.0000000000e+00, SVD da σ²_min = 1.0003319062e-08) y coinciden en float64 (C): el procedimiento de formar XᵀX es inestable en precisión baja, y esa pérdida es propiedad del procedimiento, no del problema.

## Entrega

- El notebook se ejecuta sin errores de principio a fin: una única celda de código construye los tres casos, aplica los dos procedimientos e imprime todos los resultados (además los guarda en `resultados.json`).
- Presenta los resultados de los casos A, B y C, con los espectros completos y la tabla comparativa completa (menor σ²(SVD), menor λ(XᵀX), diferencia relativa y coincidencia).
- Incluye las respuestas P1 a P5, identificadas claramente y respaldadas con la evidencia numérica de la tabla obtenida.